# 🧠 Personality Type Classification — Machine Learning Pipeline

**Author:** Julio Vergel  
**GitHub:** [juliovergel2git](https://github.com/juliovergel2git)  
**LinkedIn:** [linkedin.com/in/julio-vergel](https://linkedin.com/in/julio-vergel)

---

## Project Overview

Can we predict whether a person is an **introvert or extrovert** based on behavioural data?

This project builds a complete supervised machine learning pipeline to classify personality types using features such as time spent alone, stage fright, social event attendance, and social media activity.

Six classification algorithms are trained, evaluated, and compared — and the best-performing model's hyperparameters are further optimised using `GridSearchCV`.

**Tools:** Python · Pandas · NumPy · Scikit-learn · Matplotlib · Seaborn

---


## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 13,
                     'axes.labelsize': 11, 'font.family': 'DejaVu Sans'})

df_raw = pd.read_csv('personality_dataset.csv')
print(f"✅ Dataset loaded: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print("=" * 55)
print("DATASET OVERVIEW")
print("=" * 55)
print(f"\nColumns      : {list(df_raw.columns)}")
print(f"Numeric cols : {list(df_raw.select_dtypes(include='number').columns)}")
print(f"Categorical  : {list(df_raw.select_dtypes(include='object').columns)}")
print(f"\nClass balance:")
print(df_raw['Personality'].value_counts(normalize=True).map(lambda x: f'{x:.1%}'))
print(f"\nMissing values per column:")
print(df_raw.isnull().sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Class balance
counts = df_raw['Personality'].value_counts()
axes[0].bar(counts.index, counts.values,
            color=['cornflowerblue', 'coral'], edgecolor='white', width=0.5)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')
for i, (label, val) in enumerate(counts.items()):
    axes[0].text(i, val + 15, str(val), ha='center', fontsize=11, fontweight='bold')

# Correlation heatmap (numeric features only)
corr = df_raw.select_dtypes(include='number').corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=axes[1])
axes[1].set_title('Feature Correlation Matrix')

plt.suptitle('Dataset Overview', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('01_eda_overview.png', bbox_inches='tight')
plt.show()

In [ ]:
# Distribution of numeric features by personality type
num_features = ['Time_spent_Alone', 'Social_event_attendance',
                'Going_outside', 'Friends_circle_size', 'Post_frequency']

fig, axes = plt.subplots(1, len(num_features), figsize=(16, 4), sharey=False)

for ax, feat in zip(axes, num_features):
    for label, color in zip(['Introvert', 'Extrovert'], ['coral', 'cornflowerblue']):
        subset = df_raw[df_raw['Personality'] == label][feat].dropna()
        ax.hist(subset, bins=15, alpha=0.6, color=color, label=label, edgecolor='white')
    ax.set_title(feat.replace('_', ' '), fontsize=10)
    ax.set_xlabel('Value')

axes[0].set_ylabel('Frequency')
axes[-1].legend(loc='upper right', fontsize=9)
fig.suptitle('Feature Distributions by Personality Type', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('02_feature_distributions.png', bbox_inches='tight')
plt.show()

## 3. Data Preprocessing

In [ ]:
# 3.1 — Drop missing values
df = df_raw.dropna()
print(f"Rows after dropping NaN: {len(df):,}  (removed {len(df_raw)-len(df)} rows)")

# 3.2 — Encode categorical columns with One-Hot Encoding (drop_first to avoid multicollinearity)
df = pd.get_dummies(df,
                    columns=['Stage_fear', 'Drained_after_socializing', 'Personality'],
                    drop_first=True)

# Convert boolean columns to int for compatibility
for col in df.select_dtypes(include='bool').columns:
    df[col] = df[col].astype(int)

print(f"\nColumns after encoding: {list(df.columns)}")
print(f"\nTarget variable  →  'Personality_Introvert'  (1 = Introvert, 0 = Extrovert)")

In [ ]:
# 3.3 — Outlier removal using 3-sigma rule
original_size = len(df)

for col in df.columns:
    mean, std = df[col].mean(), df[col].std()
    if std > 0:
        df = df[(df[col] >= mean - 3*std) & (df[col] <= mean + 3*std)]

print(f"Rows before outlier removal : {original_size:,}")
print(f"Rows after outlier removal  : {len(df):,}  (removed {original_size - len(df)} rows)")
print(f"\nFinal dataset shape: {df.shape}")

## 4. Train / Test Split

In [ ]:
X = df.drop('Personality_Introvert', axis=1)
y = df['Personality_Introvert']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set   : {X_train.shape[0]:,} samples ({X_train.shape[0]/len(df):.0%})")
print(f"Test set       : {X_test.shape[0]:,} samples  ({X_test.shape[0]/len(df):.0%})")
print(f"Features used  : {list(X.columns)}")
print(f"\nClass balance in training set:")
print(y_train.value_counts(normalize=True).map(lambda x: f'{x:.1%}'))

## 5. Model Training & Comparison

Six classification algorithms are trained and evaluated. Metrics used:
- **Accuracy** — overall correct predictions
- **Recall** — ability to detect true positives (important for minority class)
- **F1-score** — harmonic mean of precision and recall


In [ ]:
models = {
    "Logistic Regression" : LogisticRegression(max_iter=500),
    "Random Forest"        : RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting"    : GradientBoostingClassifier(n_estimators=100, random_state=42),
    "SVM"                  : SVC(kernel='rbf', C=1.0),
    "KNN"                  : KNeighborsClassifier(n_neighbors=5),
    "Neural Network"       : MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, random_state=42)
}

results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred
    results[name] = {
        "Accuracy" : round(accuracy_score(y_test, y_pred), 4),
        "Recall"   : round(recall_score(y_test, y_pred, average='weighted'), 4),
        "F1-score" : round(f1_score(y_test, y_pred, average='weighted'), 4)
    }

results_df = pd.DataFrame(results).T.sort_values("Accuracy", ascending=False)
print("\n=== MODEL PERFORMANCE SUMMARY ===")
print(results_df.to_string())
print(f"\n🏆 Best model: {results_df.index[0]}  (Accuracy: {results_df['Accuracy'].iloc[0]:.2%})")

In [ ]:
# Visual comparison of all models
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['Accuracy', 'Recall', 'F1-score']
palette = sns.color_palette("muted", len(results_df))

for ax, metric in zip(axes, metrics):
    sorted_df = results_df.sort_values(metric, ascending=True)
    bars = ax.barh(sorted_df.index, sorted_df[metric],
                   color=palette, edgecolor='white', height=0.6)
    ax.set_xlim(0.85, 1.0)
    ax.set_title(metric, fontweight='bold')
    ax.set_xlabel('Score')
    for bar in bars:
        ax.text(bar.get_width() - 0.002, bar.get_y() + bar.get_height()/2,
                f'{bar.get_width():.4f}', va='center', ha='right',
                fontsize=9, color='white', fontweight='bold')

fig.suptitle('Model Comparison — All Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('03_model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# Confusion matrices for all models
fig, axes = plt.subplots(2, 3, figsize=(14, 9))
axes = axes.flatten()

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Extrovert', 'Introvert'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    acc = results[name]['Accuracy']
    ax.set_title(f'{name}\nAccuracy: {acc:.2%}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

fig.suptitle('Confusion Matrices — All Models', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('04_confusion_matrices.png', bbox_inches='tight')
plt.show()

## 6. Feature Importance Analysis

In [ ]:
rf_model = models['Random Forest']
feature_importance = pd.Series(
    rf_model.feature_importances_, index=X.columns
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['steelblue' if v < feature_importance.max() else 'coral'
          for v in feature_importance.values]
bars = ax.barh(feature_importance.index, feature_importance.values,
               color=colors, edgecolor='white', height=0.6)

ax.set_title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
ax.axvline(feature_importance.mean(), color='gray', linestyle='--',
           linewidth=1.2, label=f'Mean: {feature_importance.mean():.3f}')
ax.legend()

for bar in bars:
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('05_feature_importance.png', bbox_inches='tight')
plt.show()

print("Top 3 most predictive features:")
for feat, score in feature_importance.sort_values(ascending=False).head(3).items():
    print(f"  {feat:35s} → {score:.4f}")

## 7. Hyperparameter Optimisation — Neural Network

Since several models tied at ~92.7% accuracy, we optimise the Neural Network
using `GridSearchCV` with 5-fold cross-validation to find the best configuration.


In [ ]:
mlp_base = MLPClassifier(max_iter=300, random_state=42)

param_grid = {
    'hidden_layer_sizes' : [(50,), (100,), (50, 50)],
    'activation'         : ['relu', 'tanh'],
    'solver'             : ['adam'],
    'alpha'              : [0.0001, 0.001],
    'learning_rate'      : ['constant', 'adaptive']
}

print("Running GridSearchCV (5-fold CV)... this may take a moment.")
grid_search = GridSearchCV(
    estimator=mlp_base,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    verbose=0
)
grid_search.fit(X_train, y_train)

best_mlp   = grid_search.best_estimator_
optimised_acc = round(best_mlp.score(X_test, y_test), 4)

print(f"\n✅ Best hyperparameters found:")
for k, v in grid_search.best_params_.items():
    print(f"   {k:25s}: {v}")

print(f"\n📈 Baseline Neural Network accuracy  : {results['Neural Network']['Accuracy']:.2%}")
print(f"📈 Optimised Neural Network accuracy : {optimised_acc:.2%}")

In [ ]:
# Confusion matrix for optimised model
y_pred_opt = best_mlp.predict(X_test)
cm_opt = confusion_matrix(y_test, y_pred_opt)

fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(cm_opt, display_labels=['Extrovert', 'Introvert'])
disp.plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title(f'Optimised Neural Network\nAccuracy: {optimised_acc:.2%}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('06_optimised_nn_confusion.png', bbox_inches='tight')
plt.show()

## 8. Conclusions & Key Findings

### 🏆 Model Performance

| Model | Accuracy | F1-score |
|---|---|---|
| Logistic Regression | **92.74%** | **92.74%** |
| Gradient Boosting | **92.74%** | **92.74%** |
| SVM | **92.74%** | **92.74%** |
| Neural Network | **92.74%** | **92.74%** |
| KNN | 92.14% | 92.13% |
| Random Forest | 90.73% | 90.72% |

Four models tied at ~92.7% accuracy — a strong result for a behavioural classification task without feature engineering.

---

### 🔍 Key Findings

**1. Most predictive features (Random Forest importance):**
- `Drained_after_socializing` (24.1%) — the single most important predictor
- `Time_spent_Alone` (17.5%) — strong signal for introversion
- `Stage_fear` (17.1%) — closely linked to personality type

**2. Model selection:**
- For this dataset, **Logistic Regression and SVM** achieve the same accuracy as more complex models — suggesting the problem is largely **linearly separable**.
- Random Forest slightly underperforms, possibly because the dataset's boolean-encoded features reduce tree diversity.

**3. Hyperparameter optimisation:**
- GridSearchCV on the Neural Network confirmed the baseline architecture was already near-optimal for this dataset size.

---

### 🔄 Next Steps

- [ ] Apply **feature engineering** (interaction terms, polynomial features) to explore non-linear patterns
- [ ] Test on a **larger, real-world dataset** to validate generalisation
- [ ] Build a simple **web app with Streamlit** to predict personality type from user inputs
- [ ] Explore **SHAP values** for model interpretability

---
*Dataset: publicly available personality classification dataset. All code is original.*
